In [1]:
import json
from pathlib import Path

data_dir = Path(r"D:\project\1")
files = sorted(data_dir.glob("*.json"))
print(f"找到 {len(files)} 个 json 文件")
print(f"第一个: {files[0].name}")

with open(files[0], 'r') as f:
    data = json.load(f)

print("\n顶层 keys:", list(data.keys()))

# 看 cycles 结构
if 'cycles' in data:
    print(f"\ncycles 数量: {len(data['cycles'])}")
    print("第一个 cycle 的 keys:", list(data['cycles'][0].keys()))
    
    # 打印第一个 cycle 各字段的形状/类型
    for k, v in data['cycles'][0].items():
        if isinstance(v, list):
            print(f"  {k}: list, len={len(v)}, sample={v[:3]}")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")

找到 140 个 json 文件
第一个: FastCharge_000000_CH19_structure.json

顶层 keys: ['@module', '@class', 'barcode', 'protocol', 'channel_id', 'summary', 'cycles_interpolated', 'diagnostic_summary', 'diagnostic_interpolated', '@version']


In [2]:
import json
from pathlib import Path

data_dir = Path(r"D:\project\1")
files = sorted(data_dir.glob("*.json"))

with open(files[0], 'r') as f:
    data = json.load(f)

print("=== summary 字段 ===")
for k, v in data['summary'].items():
    if isinstance(v, list):
        print(f"  {k}: list, len={len(v)}, sample={v[:3]}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

print("\n=== cycles_interpolated 字段 ===")
for k, v in data['cycles_interpolated'].items():
    if isinstance(v, list):
        n = len(v)
        if n > 0 and isinstance(v[0], list):
            print(f"  {k}: list[{n}], 每个元素是 list, 第一个元素长度={len(v[0])}")
        else:
            print(f"  {k}: list, len={n}, sample={v[:3]}")
    else:
        print(f"  {k}: {type(v).__name__}")

=== summary 字段 ===
  cycle_index: list, len=491, sample=[0, 1, 2]
  discharge_capacity: list, len=491, sample=[1.9345720000000002, 1.0454259, 1.0480373]
  charge_capacity: list, len=491, sample=[1.4173516000000002, 1.0456483, 1.0484418]
  discharge_energy: list, len=491, sample=[6.116105999999999, 3.1736647999999996, 3.1761549]
  charge_energy: list, len=491, sample=[4.673017, 3.6461608, 3.6511858]
  dc_internal_resistance: list, len=491, sample=[0.029383694753050804, 0.017864122986793518, 0.017928939312696457]
  temperature_maximum: list, len=491, sample=[34.16896057128906, 34.85007095336914, 34.573604583740234]
  temperature_average: list, len=491, sample=[30.977693557739258, 32.641014099121094, 32.27454376220703]
  temperature_minimum: list, len=491, sample=[25.23790168762207, 30.33453941345215, 29.808897018432617]
  date_time_iso: list, len=491, sample=['2017-07-01T03:52:32+00:00', '2017-07-02T00:59:44+00:00', '2017-07-02T01:59:28+00:00']
  energy_efficiency: list, len=491, sample=

In [3]:
import json, re
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"D:\project\1")
ORIG_CSV = Path(r"D:\project\battery_master_data_final.csv")
N_CYCLES = 100

def safe_slope(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2: return 0.0
    return float(np.polyfit(np.arange(len(x)), x, 1)[0])

def extract_one(json_path):
    with open(json_path, 'r') as f:
        d = json.load(f)
    s = d['summary']
    ci = d.get('cycles_interpolated', {})
    n = min(N_CYCLES, len(s['cycle_index']))
    feats = {}

    QD = np.array(s['discharge_capacity'][:n], dtype=float)
    feats['QD_first10_mean'] = float(np.mean(QD[:10]))
    feats['QD_last10_mean']  = float(np.mean(QD[-10:]))
    feats['QD_delta']        = feats['QD_first10_mean'] - feats['QD_last10_mean']
    feats['QD_slope']        = safe_slope(QD)
    feats['QD_min']          = float(np.min(QD))
    feats['QD_std']          = float(np.std(QD))

    QC = np.array(s['charge_capacity'][:n], dtype=float)
    feats['QC_slope']        = safe_slope(QC)
    feats['QC_minus_QD']     = float(np.mean(QC - QD))

    IR = np.array(s['dc_internal_resistance'][:n], dtype=float)
    feats['IR_first10_mean'] = float(np.mean(IR[:10]))
    feats['IR_last10_mean']  = float(np.mean(IR[-10:]))
    feats['IR_delta']        = feats['IR_last10_mean'] - feats['IR_first10_mean']
    feats['IR_slope']        = safe_slope(IR)

    Tmax = np.array(s['temperature_maximum'][:n], dtype=float)
    Tavg = np.array(s['temperature_average'][:n], dtype=float)
    Tmin = np.array(s['temperature_minimum'][:n], dtype=float)
    feats['Tmax_mean']    = float(np.mean(Tmax))
    feats['Tmax_slope']   = safe_slope(Tmax)
    feats['Tmax_max']     = float(np.max(Tmax))
    feats['Tavg_mean']    = float(np.mean(Tavg))
    feats['Tavg_slope']   = safe_slope(Tavg)
    feats['T_range_mean'] = float(np.mean(Tmax - Tmin))

    CT = np.array(s['charge_duration'][:n], dtype=float)
    feats['CT_first10_mean'] = float(np.mean(CT[:10]))
    feats['CT_last10_mean']  = float(np.mean(CT[-10:]))
    feats['CT_delta']        = feats['CT_first10_mean'] - feats['CT_last10_mean']
    feats['CT_slope']        = safe_slope(CT)
    feats['CT_std']          = float(np.std(CT))

    EE = np.array(s['energy_efficiency'][:n], dtype=float)
    feats['EE_mean']  = float(np.mean(EE))
    feats['EE_slope'] = safe_slope(EE)

    V_all = ci.get('voltage', [])
    step_all = ci.get('step_type', [])
    cyc_all = ci.get('cycle_index', [])
    if len(V_all) > 0:
        V_all = np.asarray(V_all, dtype=float)
        cyc_all = np.asarray(cyc_all)
        step_all = np.asarray(step_all)
        def gdc(idx):
            mask = (cyc_all == idx) & (step_all == 'discharge')
            return V_all[mask]
        V0 = gdc(0); V99 = gdc(99) if n >= 100 else gdc(n-1)
        feats['V0_mean'] = float(V0.mean()) if len(V0) > 10 else 0.0
        feats['V0_std']  = float(V0.std())  if len(V0) > 10 else 0.0
        feats['V99_mean'] = float(V99.mean()) if len(V99) > 10 else 0.0
        feats['V_mean_shift'] = feats['V0_mean'] - feats['V99_mean']
    else:
        for k in ['V0_mean','V0_std','V99_mean','V_mean_shift']:
            feats[k] = 0.0

    feats['Q_activation'] = float(np.mean(QD[:5]) - QD[0])
    return feats

# === 批量提取 ===
files = sorted(DATA_DIR.glob("*.json"))
print(f"处理 {len(files)} 个文件...")
rows = []
for i, fp in enumerate(files):
    try:
        feats = extract_one(fp)
        feats['ID'] = fp.stem.replace('_structure', '')
        rows.append(feats)
    except Exception as e:
        print(f"✗ {fp.name}: {e}")

df_new = pd.DataFrame(rows)
print(f"\n新特征: {df_new.shape}")

# === 保存原始特征（关键：独立文件，不会被覆盖）===
RAW_CSV = r"D:\project\battery_features_raw.csv"
df_new.to_csv(RAW_CSV, index=False)
print(f"已保存原始特征: {RAW_CSV}")

# === 合并 ===
df_orig = pd.read_csv(ORIG_CSV)
df_orig['ID_clean'] = df_orig['ID'].astype(str).str.replace('.json','',regex=False).str.replace('_structure','',regex=False)
df_new['ID_clean']  = df_new['ID']

df_merged = df_new.merge(df_orig[['ID_clean','Cycle_Life','C_Rate']], on='ID_clean', how='inner')
print(f"合并后: {df_merged.shape}")

if len(df_merged) > 0:
    OUT = r"D:\project\battery_features_v2.csv"
    df_merged.to_csv(OUT, index=False)
    print(f"最终已保存: {OUT}")
    print("\n前 3 行预览:")
    print(df_merged[['ID_clean','Cycle_Life','QD_slope','IR_slope','CT_slope']].head(3))
else:
    print("merge 失败，ID 示例:")
    print("原 CSV:", df_orig['ID_clean'].head(3).tolist())
    print("新特征:", df_new['ID_clean'].head(3).tolist())

处理 140 个文件...

新特征: (140, 31)
已保存原始特征: D:\project\battery_features_raw.csv
合并后: (137, 34)
最终已保存: D:\project\battery_features_v2.csv

前 3 行预览:
                 ID_clean  Cycle_Life  QD_slope  IR_slope   CT_slope
0  FastCharge_000000_CH19         491 -0.000700 -0.000013 -19.291785
1  FastCharge_000001_CH16         667 -0.000491  0.000003 -59.909991
2  FastCharge_000001_CH30         773 -0.000451  0.000002 -59.983726


In [6]:
import pandas as pd
import numpy as np
from quantile_forest import RandomForestQuantileRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 加载
df = pd.read_csv(r"D:\project\battery_features_v2.csv")
print(f"加载: {df.shape}")

# 清洗 C_Rate=65
df = df[df['C_Rate'] <= 10].reset_index(drop=True)
print(f"清洗后: {df.shape}")

# 特征
drop_cols = ['ID', 'ID_clean', 'Cycle_Life']
feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"特征数: {len(feature_cols)}")
print(f"特征列表: {feature_cols}")

X = df[feature_cols].values
y = df['Cycle_Life'].values
y_log = np.log10(y)

# Stratified 5-Fold
y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

r2_qrf, r2_rf = [], []
mape_qrf, mae_qrf = [], []

for fold, (tr, te) in enumerate(skf.split(X, y_bins)):
    Xtr, Xte = X[tr], X[te]
    ytr_log, yte_log = y_log[tr], y_log[te]
    yte = y[te]

    # QRF
    qrf = RandomForestQuantileRegressor(n_estimators=500, random_state=42)
    qrf.fit(Xtr, ytr_log)
    p_qrf = 10 ** qrf.predict(Xte, quantiles=0.5)
    r2_qrf.append(r2_score(yte, p_qrf))
    mape_qrf.append(np.mean(np.abs((yte - p_qrf) / yte)) * 100)
    mae_qrf.append(mean_absolute_error(yte, p_qrf))

    # RF
    rf = RandomForestRegressor(n_estimators=300, max_depth=8,
                                min_samples_leaf=2, random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr_log)
    p_rf = 10 ** rf.predict(Xte)
    r2_rf.append(r2_score(yte, p_rf))

    print(f"Fold {fold+1}: QRF R²={r2_qrf[-1]:.3f}  RF R²={r2_rf[-1]:.3f}")

print("\n" + "="*50)
print(f"QRF  R²: {np.mean(r2_qrf):.4f} ± {np.std(r2_qrf):.4f}")
print(f"RF   R²: {np.mean(r2_rf):.4f} ± {np.std(r2_rf):.4f}")
print(f"QRF  MAPE: {np.mean(mape_qrf):.2f}%")
print(f"QRF  MAE : {np.mean(mae_qrf):.2f} cycles")
print("="*50)

加载: (137, 34)
清洗后: (134, 34)
特征数: 31
特征列表: ['QD_first10_mean', 'QD_last10_mean', 'QD_delta', 'QD_slope', 'QD_min', 'QD_std', 'QC_slope', 'QC_minus_QD', 'IR_first10_mean', 'IR_last10_mean', 'IR_delta', 'IR_slope', 'Tmax_mean', 'Tmax_slope', 'Tmax_max', 'Tavg_mean', 'Tavg_slope', 'T_range_mean', 'CT_first10_mean', 'CT_last10_mean', 'CT_delta', 'CT_slope', 'CT_std', 'EE_mean', 'EE_slope', 'V0_mean', 'V0_std', 'V99_mean', 'V_mean_shift', 'Q_activation', 'C_Rate']
Fold 1: QRF R²=0.769  RF R²=0.824
Fold 2: QRF R²=0.738  RF R²=0.722
Fold 3: QRF R²=0.823  RF R²=0.748
Fold 4: QRF R²=0.804  RF R²=0.749
Fold 5: QRF R²=0.754  RF R²=0.720

QRF  R²: 0.7777 ± 0.0313
RF   R²: 0.7525 ± 0.0379
QRF  MAPE: 13.18%
QRF  MAE : 98.17 cycles


In [7]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path(r"D:\project\1")
N_CYCLES = 100

# 选几个关键电压点（LFP 放电平台在 3.2-3.3V）
VOLTAGE_POINTS = [3.5, 3.45, 3.4, 3.35, 3.3, 3.25, 3.2, 3.15, 3.1]

def extract_dQ_features(json_path):
    with open(json_path, 'r') as f:
        d = json.load(f)
    s = d['summary']
    ci = d['cycles_interpolated']
    n = min(N_CYCLES, len(s['cycle_index']))
    
    feats = {}
    
    # 用 cycles_interpolated 里的原始曲线
    V = np.asarray(ci['voltage'], dtype=float)
    Qd = np.asarray(ci['discharge_capacity'], dtype=float)
    cyc = np.asarray(ci['cycle_index'])
    stp = np.asarray(ci['step_type'])
    
    # 提取第 c 个 cycle 的放电曲线 (V, Qd)
    def disch_curve(c):
        mask = (cyc == c) & (stp == 'discharge')
        return V[mask], Qd[mask]
    
    # 对第 2, 10, 50, 100 cycle 的放电曲线，在每个电压点插值出容量
    target_cycles = [2, 10, 50, 99]
    Q_at_V = {c: {} for c in target_cycles}
    
    for c in target_cycles:
        Vc, Qc = disch_curve(c)
        if len(Vc) < 10:
            for vp in VOLTAGE_POINTS:
                Q_at_V[c][vp] = np.nan
            continue
        # V 是递减的（放电），要排序
        order = np.argsort(Vc)
        Vs, Qs = Vc[order], Qc[order]
        for vp in VOLTAGE_POINTS:
            if Vs.min() <= vp <= Vs.max():
                Q_at_V[c][vp] = float(np.interp(vp, Vs, Qs))
            else:
                Q_at_V[c][vp] = np.nan
    
    # ★ 核心特征：ΔQ(V) = Q_{cycle 2}(V) - Q_{cycle 99}(V)
    # 这是 Severson 论文中最强的特征
    for vp in VOLTAGE_POINTS:
        q2 = Q_at_V[2][vp]
        q99 = Q_at_V[99][vp]
        feats[f'dQ_{vp}V'] = float(q2 - q99) if (not np.isnan(q2) and not np.isnan(q99)) else 0.0
    
    # 各 cycle 的 Q(V) 斜率（电压平台特征）
    for c in target_cycles:
        qs = [Q_at_V[c][vp] for vp in VOLTAGE_POINTS]
        qs = [q for q in qs if not np.isnan(q)]
        if len(qs) >= 3:
            feats[f'Q_slope_cyc{c}'] = float(np.polyfit(range(len(qs)), qs, 1)[0])
        else:
            feats[f'Q_slope_cyc{c}'] = 0.0
    
    # 温度积分（热应力累积）
    TTI = np.array(s.get('time_temperature_integrated', [0]*n)[:n], dtype=float)
    feats['TTI_first10'] = float(np.mean(TTI[:10]))
    feats['TTI_last10']  = float(np.mean(TTI[-10:]))
    feats['TTI_slope']   = float(np.polyfit(np.arange(n), TTI, 1)[0]) if n > 1 else 0.0
    
    # 能量效率斜率（老化指标）
    EE = np.array(s['energy_efficiency'][:n], dtype=float)
    feats['EE_last10']  = float(np.mean(EE[-10:]))
    feats['EE_delta']   = float(np.mean(EE[:10]) - np.mean(EE[-10:]))
    
    # 库仑效率（充电容量 / 放电容量）
    Qc = np.array(s['charge_capacity'][:n], dtype=float)
    Qd_summary = np.array(s['discharge_capacity'][:n], dtype=float)
    CE = Qd_summary / np.maximum(Qc, 1e-6)
    feats['CE_mean']  = float(np.mean(CE))
    feats['CE_slope'] = float(np.polyfit(np.arange(n), CE, 1)[0]) if n > 1 else 0.0
    
    return feats

# 批量
files = sorted(DATA_DIR.glob("*.json"))
print(f"提取 ΔQ(V) 特征，共 {len(files)} 个文件...")

rows = []
for i, fp in enumerate(files):
    try:
        feats = extract_dQ_features(fp)
        feats['ID'] = fp.stem.replace('_structure', '')
        rows.append(feats)
    except Exception as e:
        print(f"✗ {fp.name}: {e}")

df_dq = pd.DataFrame(rows)
print(f"新特征表: {df_dq.shape}")

# 合并到主表
df_main = pd.read_csv(r"D:\project\battery_features_v2.csv")
df_main['ID_key'] = df_main['ID_clean'].str.replace('.json','',regex=False).str.replace('_structure','',regex=False)
df_dq['ID_key'] = df_dq['ID']

df_combined = df_main.merge(df_dq.drop(columns=['ID']), on='ID_key', how='inner')
print(f"合并后: {df_combined.shape}")

df_combined.to_csv(r"D:\project\battery_features_v3.csv", index=False)
print(f"已保存: battery_features_v3.csv")
print(f"总特征数: {len([c for c in df_combined.columns if c not in ['ID','ID_clean','ID_key','Cycle_Life','C_Rate']])}")

提取 ΔQ(V) 特征，共 140 个文件...
✗ FastCharge_000001_CH16_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000001_CH38_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000002_CH10_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000002_CH2_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000002_CH47_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000006_CH19_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000006_CH27_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000007_CH24_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000007_CH39_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000012_CH13_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000012_CH15_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000012_CH23_struc

In [8]:
import pandas as pd
import numpy as np
from quantile_forest import RandomForestQuantileRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r"D:\project\battery_features_v3.csv")
df = df[df['C_Rate'] <= 10].reset_index(drop=True)

drop_cols = ['ID', 'ID_clean', 'ID_key', 'Cycle_Life']
feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"样本: {len(df)}, 特征: {len(feature_cols)}")

X = df[feature_cols].values
y = df['Cycle_Life'].values
y_log = np.log10(y)

y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

r2_qrf, mape_qrf, mae_qrf = [], [], []

for fold, (tr, te) in enumerate(skf.split(X, y_bins)):
    qrf = RandomForestQuantileRegressor(n_estimators=500, random_state=42)
    qrf.fit(X[tr], y_log[tr])
    p = 10 ** qrf.predict(X[te], quantiles=0.5)
    r2 = r2_score(y[te], p)
    r2_qrf.append(r2)
    mape_qrf.append(np.mean(np.abs((y[te] - p) / y[te])) * 100)
    mae_qrf.append(mean_absolute_error(y[te], p))
    print(f"Fold {fold+1}: R²={r2:.3f}")

print("\n" + "="*50)
print(f"QRF R²: {np.mean(r2_qrf):.4f} ± {np.std(r2_qrf):.4f}")
print(f"MAPE:   {np.mean(mape_qrf):.2f}%")
print(f"MAE:    {np.mean(mae_qrf):.2f} cycles")
print("="*50)

样本: 111, 特征: 51
Fold 1: R²=0.821
Fold 2: R²=0.841
Fold 3: R²=0.838
Fold 4: R²=0.891
Fold 5: R²=0.851

QRF R²: 0.8481 ± 0.0235
MAPE:   11.14%
MAE:    75.76 cycles


In [9]:
import pandas as pd

df_main = pd.read_csv(r"D:\project\battery_features_v2.csv")
df_dq   = pd.read_csv(r"D:\project\battery_features_raw.csv")  # 看原始提取结果

# v2 有多少
print(f"v2: {df_main.shape}")

# 原始提取表
print(f"raw: {df_dq.shape}")

# v3（合并后）
df_v3 = pd.read_csv(r"D:\project\battery_features_v3.csv")
print(f"v3: {df_v3.shape}")

# 对比 ID
ids_v2 = set(df_main['ID_clean'].astype(str))
ids_v3 = set(df_v3['ID_clean'].astype(str))
lost = ids_v2 - ids_v3
print(f"\nv2 有但 v3 丢了的 cell: {len(lost)} 个")
print(f"示例: {list(lost)[:10]}")

v2: (137, 34)
raw: (140, 31)
v3: (114, 55)

v2 有但 v3 丢了的 cell: 23 个
示例: ['FastCharge_000050_CH40', 'FastCharge_000002_CH47', 'FastCharge_000015_CH14', 'FastCharge_000007_CH24', 'FastCharge_000012_CH45', 'FastCharge_000012_CH37', 'FastCharge_000002_CH10', 'FastCharge_000001_CH38', 'FastCharge_000017_CH46', 'FastCharge_000006_CH19']


In [10]:
import pandas as pd

df_main = pd.read_csv(r"D:\project\battery_features_v2.csv")
df_dq   = pd.read_csv(r"D:\project\battery_features_raw.csv")

# 清理 ID
df_main['key'] = df_main['ID_clean'].astype(str).str.strip().str.replace('.json','',regex=False).str.replace('_structure','',regex=False)
df_dq['key']   = df_dq['ID'].astype(str).str.strip().str.replace('.json','',regex=False).str.replace('_structure','',regex=False)

main_keys = set(df_main['key'])
raw_keys  = set(df_dq['key'])

print(f"main: {len(main_keys)}   raw: {len(raw_keys)}   交集: {len(main_keys & raw_keys)}")
print(f"\nmain - raw  (v2有但raw没有): {list(main_keys - raw_keys)[:10]}")
print(f"raw - main  (raw有但v2没有): {list(raw_keys - main_keys)[:10]}")

# 用干净的 key 重新 merge
if 'ID' in df_dq.columns:
    df_dq = df_dq.drop(columns=['ID'])
if 'ID_key' in df_main.columns:
    df_main = df_main.drop(columns=['ID_key'])
if 'ID_key' in df_dq.columns:
    df_dq = df_dq.drop(columns=['ID_key'])

df_combined = df_main.merge(df_dq, on='key', how='inner')
print(f"\n合并后: {df_combined.shape}")
df_combined.to_csv(r"D:\project\battery_features_v3.csv", index=False)
print("已保存: battery_features_v3.csv")

main: 137   raw: 140   交集: 137

main - raw  (v2有但raw没有): []
raw - main  (raw有但v2没有): ['FastCharge_000026_CH6', 'FastCharge_000002_CH26', 'FastCharge_000004_CH2']

合并后: (137, 65)
已保存: battery_features_v3.csv


In [13]:
import pandas as pd
df = pd.read_csv(r"D:\project\battery_features_v3.csv")
print(f"shape: {df.shape}")
print(f"\n所有列名:")
for i, c in enumerate(df.columns):
    print(f"  {i:3d}: {c}")

shape: (137, 65)

所有列名:
    0: QD_first10_mean_x
    1: QD_last10_mean_x
    2: QD_delta_x
    3: QD_slope_x
    4: QD_min_x
    5: QD_std_x
    6: QC_slope_x
    7: QC_minus_QD_x
    8: IR_first10_mean_x
    9: IR_last10_mean_x
   10: IR_delta_x
   11: IR_slope_x
   12: Tmax_mean_x
   13: Tmax_slope_x
   14: Tmax_max_x
   15: Tavg_mean_x
   16: Tavg_slope_x
   17: T_range_mean_x
   18: CT_first10_mean_x
   19: CT_last10_mean_x
   20: CT_delta_x
   21: CT_slope_x
   22: CT_std_x
   23: EE_mean_x
   24: EE_slope_x
   25: V0_mean_x
   26: V0_std_x
   27: V99_mean_x
   28: V_mean_shift_x
   29: Q_activation_x
   30: ID
   31: ID_clean
   32: Cycle_Life
   33: C_Rate
   34: key
   35: QD_first10_mean_y
   36: QD_last10_mean_y
   37: QD_delta_y
   38: QD_slope_y
   39: QD_min_y
   40: QD_std_y
   41: QC_slope_y
   42: QC_minus_QD_y
   43: IR_first10_mean_y
   44: IR_last10_mean_y
   45: IR_delta_y
   46: IR_slope_y
   47: Tmax_mean_y
   48: Tmax_slope_y
   49: Tmax_max_y
   50: Tavg_mean_y


In [1]:
import pandas as pd
import numpy as np
from quantile_forest import RandomForestQuantileRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# === 用 v2（137 cells, 31 特征）===
df = pd.read_csv(r"D:\project\battery_features_v2.csv")
df = df[df['C_Rate'] <= 10].reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in ['ID', 'ID_clean', 'Cycle_Life']]
print(f"样本: {len(df)}, 特征: {len(feature_cols)}")
print(f"特征: {feature_cols}")

X = df[feature_cols].values
y = df['Cycle_Life'].values
y_log = np.log10(y)

y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

r2s, mapes, maes, rmses, picps, mpiws, pinaws = [], [], [], [], [], [], []
all_y, all_p, all_lo, all_hi = [], [], [], []

for fold, (tr, te) in enumerate(skf.split(X, y_bins)):
    qrf = RandomForestQuantileRegressor(n_estimators=500, random_state=42)
    qrf.fit(X[tr], y_log[tr])
    p_med = 10 ** qrf.predict(X[te], quantiles=0.5)
    p_lo  = 10 ** qrf.predict(X[te], quantiles=0.05)
    p_hi  = 10 ** qrf.predict(X[te], quantiles=0.95)
    yte = y[te]
    all_y.extend(yte); all_p.extend(p_med)
    all_lo.extend(p_lo); all_hi.extend(p_hi)

    r2s.append(r2_score(yte, p_med))
    mapes.append(np.mean(np.abs((yte - p_med) / yte)) * 100)
    maes.append(mean_absolute_error(yte, p_med))
    rmses.append(np.sqrt(mean_squared_error(yte, p_med)))
    picps.append(np.mean((yte >= p_lo) & (yte <= p_hi)) * 100)
    mpiws.append(np.mean(p_hi - p_lo))
    pinaws.append(np.mean(p_hi - p_lo) / (yte.max() - yte.min()))

print("\n" + "=" * 55)
print(f"R²     : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
print(f"MAPE   : {np.mean(mapes):.2f}%")
print(f"MAE    : {np.mean(maes):.2f} cycles")
print(f"RMSE   : {np.mean(rmses):.2f} cycles")
print(f"PICP   : {np.mean(picps):.2f}%")
print(f"MPIW   : {np.mean(mpiws):.2f} cycles")
print(f"PINAW  : {np.mean(pinaws):.4f}")
print("=" * 55)

# 保存结果供论文用
results = pd.DataFrame({
    'actual': all_y, 'pred_median': all_p,
    'pred_lower': all_lo, 'pred_upper': all_hi
})
results.to_csv(r"D:\project\final_predictions.csv", index=False)
print(f"\n预测结果已保存: final_predictions.csv")

样本: 134, 特征: 31
特征: ['QD_first10_mean', 'QD_last10_mean', 'QD_delta', 'QD_slope', 'QD_min', 'QD_std', 'QC_slope', 'QC_minus_QD', 'IR_first10_mean', 'IR_last10_mean', 'IR_delta', 'IR_slope', 'Tmax_mean', 'Tmax_slope', 'Tmax_max', 'Tavg_mean', 'Tavg_slope', 'T_range_mean', 'CT_first10_mean', 'CT_last10_mean', 'CT_delta', 'CT_slope', 'CT_std', 'EE_mean', 'EE_slope', 'V0_mean', 'V0_std', 'V99_mean', 'V_mean_shift', 'Q_activation', 'C_Rate']

R²     : 0.7777 ± 0.0313
MAPE   : 13.18%
MAE    : 98.17 cycles
RMSE   : 166.23 cycles
PICP   : 85.78%
MPIW   : 436.40 cycles
PINAW  : 0.3108

预测结果已保存: final_predictions.csv


In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"D:\project\1")
N_CYCLES = 100
VOLTAGE_POINTS = [3.5, 3.45, 3.4, 3.35, 3.3, 3.25, 3.2, 3.15, 3.1]

def extract_dQ(json_path):
    with open(json_path, 'r') as f:
        d = json.load(f)
    s = d['summary']
    ci = d['cycles_interpolated']
    n = min(N_CYCLES, len(s['cycle_index']))

    feats = {}
    V   = np.asarray(ci['voltage'], dtype=float)
    Qd  = np.asarray(ci['discharge_capacity'], dtype=float)
    cyc = np.asarray(ci['cycle_index'])
    stp = np.asarray(ci['step_type'])

    def disch_curve(c):
        mask = (cyc == c) & (stp == 'discharge')
        return V[mask], Qd[mask]

    target_cycles = [2, 10, 50, 99]
    Q_at_V = {}
    for c in target_cycles:
        Vc, Qc = disch_curve(c)
        Q_at_V[c] = {}
        if len(Vc) < 10:
            for vp in VOLTAGE_POINTS:
                Q_at_V[c][vp] = np.nan
            continue
        order = np.argsort(Vc)
        Vs, Qs = Vc[order], Qc[order]
        for vp in VOLTAGE_POINTS:
            if Vs.min() <= vp <= Vs.max():
                Q_at_V[c][vp] = float(np.interp(vp, Vs, Qs))
            else:
                Q_at_V[c][vp] = np.nan

    # ★ 核心：ΔQ(V) = Q_cyc2(V) - Q_cyc99(V)
    for vp in VOLTAGE_POINTS:
        q2, q99 = Q_at_V[2][vp], Q_at_V[99][vp]
        feats[f'dQ_{vp}V'] = float(q2 - q99) if (not np.isnan(q2) and not np.isnan(q99)) else 0.0

    # Q(V) 曲线斜率
    for c in target_cycles:
        qs = [Q_at_V[c][vp] for vp in VOLTAGE_POINTS if not np.isnan(Q_at_V[c][vp])]
        feats[f'Q_slope_cyc{c}'] = float(np.polyfit(range(len(qs)), qs, 1)[0]) if len(qs) >= 3 else 0.0

    # TTI
    TTI = np.array(s.get('time_temperature_integrated', [0]*n)[:n], dtype=float)
    feats['TTI_first10'] = float(np.mean(TTI[:10]))
    feats['TTI_last10']  = float(np.mean(TTI[-10:]))
    feats['TTI_slope']   = float(np.polyfit(np.arange(n), TTI, 1)[0]) if n > 1 else 0.0

    # EE
    EE = np.array(s['energy_efficiency'][:n], dtype=float)
    feats['EE_last10'] = float(np.mean(EE[-10:]))
    feats['EE_delta']  = float(np.mean(EE[:10]) - np.mean(EE[-10:]))

    # CE
    Qc   = np.array(s['charge_capacity'][:n], dtype=float)
    Qd_s = np.array(s['discharge_capacity'][:n], dtype=float)
    CE   = Qd_s / np.maximum(Qc, 1e-6)
    feats['CE_mean']  = float(np.mean(CE))
    feats['CE_slope'] = float(np.polyfit(np.arange(n), CE, 1)[0]) if n > 1 else 0.0

    return feats

files = sorted(DATA_DIR.glob("*.json"))
print(f"提取 dQ 特征，共 {len(files)} 个文件...")
rows = []
for i, fp in enumerate(files):
    try:
        feats = extract_dQ(fp)
        feats['ID_clean'] = fp.stem.replace('_structure', '')
        rows.append(feats)
    except Exception as e:
        print(f"✗ {fp.name}: {e}")

df_dq = pd.DataFrame(rows)
print(f"dQ 特征表: {df_dq.shape}")
print(f"dQ 列名: {[c for c in df_dq.columns if c.startswith('dQ_')]}")
df_dq.to_csv(r"D:\project\battery_dq_features.csv", index=False)
print("✓ 已保存: battery_dq_features.csv")

提取 dQ 特征，共 140 个文件...
✗ FastCharge_000001_CH16_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000001_CH38_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000002_CH10_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000002_CH2_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000002_CH47_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000006_CH19_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000006_CH27_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000007_CH24_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000007_CH39_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000012_CH13_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000012_CH15_structure.json: SVD did not converge in Linear Least Squares
✗ FastCharge_000012_CH23_structur

In [4]:
import pandas as pd

df_v2 = pd.read_csv(r"D:\project\battery_features_v2.csv")
df_dq = pd.read_csv(r"D:\project\battery_dq_features.csv")

# 清理可能存在的旧 key 列
df_v2 = df_v2.drop(columns=[c for c in ['key','ID_key'] if c in df_v2.columns], errors='ignore')
df_dq = df_dq.drop(columns=[c for c in ['key','ID_key'] if c in df_dq.columns], errors='ignore')

# 统一 key
df_v2['key'] = df_v2['ID_clean'].astype(str).str.strip()
df_dq['key'] = df_dq['ID_clean'].astype(str).str.strip()

print(f"v2: {df_v2.shape}, dQ: {df_dq.shape}")
print(f"ID 交集: {len(set(df_v2['key']) & set(df_dq['key']))}")

# 合并（列名不重叠，不会有 _x/_y）
df_v3 = df_v2.merge(df_dq.drop(columns=['ID_clean']), on='key', how='inner')
print(f"合并后: {df_v3.shape}")

# 检查 _x/_y
xy = [c for c in df_v3.columns if c.endswith('_x') or c.endswith('_y')]
print(f"_x/_y 后缀列: {len(xy)}")
print(f"dQ 特征是否在: {[c for c in df_v3.columns if c.startswith('dQ_')][:5]}")

df_v3.to_csv(r"D:\project\battery_features_v3.csv", index=False)
print("✓ 已保存: battery_features_v3.csv")

v2: (137, 35), dQ: (117, 22)
ID 交集: 114
合并后: (114, 55)
_x/_y 后缀列: 0
dQ 特征是否在: ['dQ_3.5V', 'dQ_3.45V', 'dQ_3.4V', 'dQ_3.35V', 'dQ_3.3V']
✓ 已保存: battery_features_v3.csv


In [5]:
import pandas as pd
import numpy as np
from quantile_forest import RandomForestQuantileRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r"D:\project\battery_features_v3.csv")
df = df[df['C_Rate'] <= 10].reset_index(drop=True)

drop_cols = ['ID', 'ID_clean', 'key', 'Cycle_Life']
feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"样本: {len(df)}, 特征: {len(feature_cols)}")

X = df[feature_cols].values
y = df['Cycle_Life'].values
y_log = np.log10(y)

y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

r2s, mapes, maes, rmses, picps, mpiws, pinaws = [], [], [], [], [], [], []
all_y, all_p, all_lo, all_hi = [], [], [], []

for fold, (tr, te) in enumerate(skf.split(X, y_bins)):
    qrf = RandomForestQuantileRegressor(n_estimators=500, random_state=42)
    qrf.fit(X[tr], y_log[tr])
    p_med = 10 ** qrf.predict(X[te], quantiles=0.5)
    p_lo  = 10 ** qrf.predict(X[te], quantiles=0.05)
    p_hi  = 10 ** qrf.predict(X[te], quantiles=0.95)
    yte = y[te]
    all_y.extend(yte); all_p.extend(p_med); all_lo.extend(p_lo); all_hi.extend(p_hi)

    r2s.append(r2_score(yte, p_med))
    mapes.append(np.mean(np.abs((yte - p_med) / yte)) * 100)
    maes.append(mean_absolute_error(yte, p_med))
    rmses.append(np.sqrt(mean_squared_error(yte, p_med)))
    picps.append(np.mean((yte >= p_lo) & (yte <= p_hi)) * 100)
    mpiws.append(np.mean(p_hi - p_lo))
    pinaws.append(np.mean(p_hi - p_lo) / (yte.max() - yte.min()))
    print(f"Fold {fold+1}: R²={r2s[-1]:.4f}  PICP={picps[-1]:.2f}%")

print("\n" + "=" * 55)
print(f"R²     : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
print(f"MAPE   : {np.mean(mapes):.2f}%")
print(f"MAE    : {np.mean(maes):.2f} cycles")
print(f"RMSE   : {np.mean(rmses):.2f} cycles")
print(f"PICP   : {np.mean(picps):.2f}%")
print(f"MPIW   : {np.mean(mpiws):.2f} cycles")
print("=" * 55)

# 保存预测结果供论文用
pd.DataFrame({
    'actual': all_y, 'pred_median': all_p,
    'pred_lower': all_lo, 'pred_upper': all_hi
}).to_csv(r"D:\project\final_predictions.csv", index=False)
print("✓ 预测结果已保存")

样本: 111, 特征: 51
Fold 1: R²=0.8206  PICP=100.00%
Fold 2: R²=0.8405  PICP=81.82%
Fold 3: R²=0.8378  PICP=90.91%
Fold 4: R²=0.8910  PICP=90.91%
Fold 5: R²=0.8506  PICP=81.82%

R²     : 0.8481 ± 0.0235
MAPE   : 11.14%
MAE    : 75.76 cycles
RMSE   : 124.56 cycles
PICP   : 89.09%
MPIW   : 394.24 cycles
✓ 预测结果已保存


In [6]:
import pandas as pd

df_v2 = pd.read_csv(r"D:\project\battery_features_v2.csv")
df_dq = pd.read_csv(r"D:\project\battery_dq_features.csv")

print(f"v2: {df_v2.shape}")
print(f"dQ: {df_dq.shape}")

# 看两边 ID
v2_ids = set(df_v2['ID_clean'].astype(str).str.strip())
dq_ids = set(df_dq['ID_clean'].astype(str).str.strip())

print(f"\nv2 唯一 ID: {len(v2_ids)}")
print(f"dQ 唯一 ID: {len(dq_ids)}")
print(f"交集: {len(v2_ids & dq_ids)}")

lost = v2_ids - dq_ids
print(f"\nv2 有但 dQ 没有的 cell: {len(lost)} 个")
print(f"示例: {sorted(list(lost))[:5]}")

# 关键：dQ 表里这些 cell 是不是特征全为 0？
if len(lost) > 0:
    lost_in_dq = df_dq[df_dq['ID_clean'].isin(lost)]
    print(f"\ndQ 表里这些 cell 的行数: {len(lost_in_dq)}")

v2: (137, 34)
dQ: (117, 21)

v2 唯一 ID: 137
dQ 唯一 ID: 117
交集: 114

v2 有但 dQ 没有的 cell: 23 个
示例: ['FastCharge_000001_CH16', 'FastCharge_000001_CH38', 'FastCharge_000002_CH10', 'FastCharge_000002_CH2', 'FastCharge_000002_CH47']

dQ 表里这些 cell 的行数: 0
